In [1]:
# main.py

import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import cv2
from ultralytics import YOLO
import numpy as np
import itertools
from tkinter import Tk
from tkinter.filedialog import askopenfilename
import sys

# ---------- 選擇模型與影片 ----------
MODEL_PATH = "modelv1.pt"
Tk().withdraw()
VIDEO_PATH = askopenfilename(title="請選擇影片檔案", filetypes=[("MP4 files", "*.mp4"), ("All files", "*.*")])
if not VIDEO_PATH:
    print("未選擇影片，程式結束。")
    sys.exit()

# ---------- 畫選取區域 ----------
drawing = False
areas = []
temp = []

def draw_rect(event, x, y, flags, param):
    global drawing, temp, areas
    if event == cv2.EVENT_LBUTTONDOWN:
        drawing = True
        temp = [(x, y)]
    elif event == cv2.EVENT_LBUTTONUP:
        drawing = False
        temp.append((x, y))
        if len(temp) == 2:
            areas.append(tuple(temp))
            temp = []

cap = cv2.VideoCapture(VIDEO_PATH)
ret, first_frame = cap.read()
if not ret:
    print("無法讀取影片")
    sys.exit()

cv2.namedWindow("Draw Areas")
cv2.setMouseCallback("Draw Areas", draw_rect)
while True:
    disp = first_frame.copy()
    for rect in areas:
        cv2.rectangle(disp, rect[0], rect[1], (0, 255, 0), 2)
    if len(temp) == 2:
        cv2.rectangle(disp, temp[0], temp[1], (0, 0, 255), 2)
    cv2.imshow("Draw Areas", disp)
    key = cv2.waitKey(1) & 0xFF
    if key == 27:
        cap.release()
        cv2.destroyAllWindows()
        sys.exit()
    if key == ord('c'):
        if temp:
            temp.clear()
        elif areas:
            areas.pop()
    if key == 13 and len(areas) >= 1:
        break
cv2.destroyWindow("Draw Areas")

# ---------- 初始化 YOLO ----------
model = YOLO(MODEL_PATH)
names = model.names
palette = [(255,0,0),(0,255,0),(0,0,255),(255,255,0),(255,0,255),
           (0,255,255),(128,128,0),(128,0,128),(0,128,128),(255,255,255)]

# ---------- 初始化統計 ----------
counted_ids = {
    'motor': set(),
    'car': set(),
    'truck': set(),
    'bus': set()
}
vehicle_counts = {
    'motor': 0,
    'car': 0,
    'truck': 0,
    'bus': 0
}

# ---------- 判斷重疊 ----------
def is_overlap(box1, box2):
    ax1, ay1, ax2, ay2 = box1
    bx1, by1, bx2, by2 = box2
    inter_x1 = max(ax1, bx1)
    inter_y1 = max(ay1, by1)
    inter_x2 = min(ax2, bx2)
    inter_y2 = min(ay2, by2)
    return inter_x1 < inter_x2 and inter_y1 < inter_y2

# ---------- 主迴圈 ----------
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    results = model.track(frame, persist=True, tracker="bytetrack.yaml")
    boxes = results[0].boxes

    if boxes is not None:
        ids = boxes.id.cpu().numpy() if boxes.id is not None else itertools.count()
        classes = boxes.cls.cpu().numpy()
        confs = boxes.conf.cpu().numpy()

        for box, tid, cls, conf in zip(boxes.xyxy.cpu().numpy(), ids, classes, confs):
            tid = int(tid)
            class_name = names[int(cls)]
            x1, y1, x2, y2 = map(int, box)
            det_box = (x1, y1, x2, y2)
            color = palette[int(cls) % len(palette)]

            # 畫框與標籤
            label = f"{class_name} {conf:.2f}"
            cv2.rectangle(frame, (x1,y1), (x2,y2), color, 2)
            cv2.putText(frame, label, (x1, y1-8), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

            if class_name in vehicle_counts:
                for area in areas:
                    area_box = (
                        min(area[0][0], area[1][0]),
                        min(area[0][1], area[1][1]),
                        max(area[0][0], area[1][0]),
                        max(area[0][1], area[1][1])
                    )
                    if is_overlap(det_box, area_box) and tid not in counted_ids[class_name]:
                        vehicle_counts[class_name] += 1
                        counted_ids[class_name].add(tid)
                        break

    # 畫區域
    for rect in areas:
        cv2.rectangle(frame, rect[0], rect[1], (0, 255, 0), 2)

    # 顯示結果
    y_offset = 40
    for i, cls_name in enumerate(['motor', 'car', 'truck', 'bus']):
        count = vehicle_counts.get(cls_name, 0)
        cv2.putText(frame, f"{cls_name.capitalize()} passed: {count}",
                    (30, y_offset + i * 40),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 255), 3)

    cv2.imshow("YOLOv8 Crossing Counter", frame)
    if cv2.waitKey(1) & 0xFF == 27:
        break

cap.release()
cv2.destroyAllWindows()


2025-07-08 14:53:19.126 python[12904:284246] The class 'NSOpenPanel' overrides the method identifier.  This method is implemented by class 'NSWindow'



0: 384x640 15 cars, 1 truck, 77.4ms
Speed: 2.0ms preprocess, 77.4ms inference, 1.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 15 cars, 1 truck, 71.3ms
Speed: 1.2ms preprocess, 71.3ms inference, 0.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 15 cars, 1 truck, 69.6ms
Speed: 1.4ms preprocess, 69.6ms inference, 0.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 15 cars, 1 truck, 70.0ms
Speed: 1.3ms preprocess, 70.0ms inference, 0.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 14 cars, 1 truck, 72.2ms
Speed: 1.3ms preprocess, 72.2ms inference, 0.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 14 cars, 1 truck, 71.4ms
Speed: 1.1ms preprocess, 71.4ms inference, 0.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 14 cars, 1 truck, 71.8ms
Speed: 1.2ms preprocess, 71.8ms inference, 0.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 14 cars, 1 truck, 72.4ms
Speed: 1.2ms preprocess, 

: 